In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
import sys, os
sys.path.append('..')

# 原始数据


In [ ]:
raw_data_path = "data/raw_data"
gdb_output_path = "data/gdb_data"

In [ ]:
import zipfile
import shutil

# 遍历 raw_data_path 下的所有文件，将每个文件解压后，再遍历解压后的文件夹的所有文件，将每个文件接下后的.gdb文件保存至data/gdb_data目录下

os.makedirs(gdb_output_path, exist_ok=True)

for file in os.listdir(raw_data_path):
    file_path = os.path.join(raw_data_path, file)
    if zipfile.is_zipfile(file_path):
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            extract_path = os.path.join(raw_data_path,
                                        os.path.splitext(file)[0])
            zip_ref.extractall(extract_path)
            # 二次遍历解压
            for root, dirs, files in os.walk(extract_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    if zipfile.is_zipfile(file_path):
                        with zipfile.ZipFile(file_path, 'r') as zip_ref_inner:
                            inner_extract_path = os.path.join(root, os.path.splitext(file)[0])
                            zip_ref_inner.extractall(inner_extract_path)
                # 遍历解压后的文件夹，找到 .gdb 文件夹
                for root, dirs, files in os.walk(extract_path):
                    for d in dirs:
                        if d.endswith('.gdb'):
                            src = os.path.join(root, d)
                            dst = os.path.join(gdb_output_path, d)
                            if not os.path.exists(dst):
                                # 检查源文件夹内是否存在同名的子文件夹（双层嵌套）
                                inner_gdb = os.path.join(src, d)
                                if os.path.isdir(inner_gdb):
                                    # 如果存在双层嵌套，直接复制内层的文件夹
                                    shutil.copytree(inner_gdb, dst)
                                    print(f"已保存（从嵌套路径）: {dst}")
                                else:
                                    # 否则直接复制源文件夹
                                    shutil.copytree(src, dst)
                                    print(f"已保存: {dst}")
                            else:
                                print(f"已存在，跳过: {dst}")

# 提取水系数据

In [ ]:
# 遍历gdb_output_path目录下的所有文件，提取每个文件中layer='HYDL'的数据，合并为一个新的df
df_all = gpd.GeoDataFrame()
for file in os.listdir(gdb_output_path):
    if file.endswith('.gdb'):
        file_path = os.path.join(gdb_output_path, file)
        df = gpd.read_file(file_path, layer='HYDL', engine='pyogrio')
        df_all = pd.concat([df_all, df], ignore_index=True)


In [ ]:
# 将df_all 保存为 river_raw.csv文件
df_all.to_csv('river_raw.csv', index=False)

# 测试

In [3]:
df_test = gpd.read_file("data/gdb_data/J49.gdb", layer='HYDL', engine='pyogrio')

In [4]:
df_test

,GB,HYDC,NAME,PERIOD,SHAPE_Length,geometry
0,210101,None,None,None,1.214176e-13,"MULTILINESTRING ((109.29644 36, 109.29644 36))"
1,210101,None,None,None,1.340649e-13,"MULTILINESTRING ((110.85186 36, 110.85186 36))"
2,210300,DB999999,小峪河,None,1.421085e-13,"MULTILINESTRING ((111.15046 36, 111.15046 36))"
3,210101,None,None,None,2.799214e-13,"MULTILINESTRING ((110.75905 36, 110.75905 36))"
4,220300,DC99999Q,汾西灌区七一渠,None,1.271057e-13,"MULTILINESTRING ((111.31528 36, 111.31528 36))"
...,...,...,...,...,...,...
2905,210400,DB210005,秃尾河,None,1.403773e-02,"MULTILINESTRING ((110.11375 38.73336, 110.1166..."
2906,210101,DC140005,文峪河,None,6.249749e-03,"MULTILINESTRING ((111.80136 37.62089, 111.8014..."
2907,210101,DC999999,西葫芦河,None,1.779787e-02,"MULTILINESTRING ((111.77747 37.676, 111.78182 ..."
2908,210400,DC140005,文峪河,None,4.810980e-02,"MULTILINESTRING ((111.75452 37.65372, 111.7552..."


In [ ]:
df_all[df_all['NAME'] == '东风渠']['geometry'].head(1).values

In [ ]:
df_all[df_all['NAME'] == '长江']

In [ ]:
df_all[df_all['HYDC'] == 'FA040001']

In [ ]:
df_all[df_all['HYDC'] == 'FB010001']

In [ ]:
df_all[df_all['HYDC'] == 'HE999999']

In [ ]:

gdb_path = f"{gdb_output_path}/E50.gdb/E50.gdb"  # 指向整个文件夹

# 1. 先列出所有图层名称
layers = gpd.list_layers(gdb_path)
print(layers)

In [ ]:
# 2. 读取特定图层
df = gpd.read_file(gdb_path, layer='HYDL', engine='pyogrio')

# 3. 查看数据（前5行）
df['length_m'] = df['SHAPE_Length'] * 1000000
# 将length_m由科学计数转为一般数字
df['length_m'] = df['length_m'].apply(lambda x: format(x, 'f'))
df.sort_values(by='length_m', ascending=False)

In [ ]:
df.info()

In [ ]:

# 4. 如果是河流线数据，提取起点（发源地）坐标
# 注意：geometry 是矢量数据的核心列
if df.geometry.type[0] == 'LineString':
    df['start_point'] = df.geometry.apply(lambda x: (x.coords[0])) # 提取经纬度元组
    print("已成功提取发源地坐标！")